In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet50
import pandas as pd
import numpy as np
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, 
                             confusion_matrix, roc_auc_score, average_precision_score,
                             cohen_kappa_score, matthews_corrcoef, roc_curve, 
                             precision_recall_curve, label_ranking_average_precision_score)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ==================== Dataset Class ====================
class MultiLabelDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.FloatTensor(self.labels[idx])
        return image, label

# ==================== Data Loading ====================
def load_dataset(root_dir, labels_dir):
    """Load dataset from folder structure and CSV labels"""
    all_image_paths = []
    all_labels = []
    
    # Get label names from first CSV file
    csv_files = [f for f in os.listdir(labels_dir) if f.endswith('.csv')]
    first_csv = pd.read_csv(os.path.join(labels_dir, csv_files[0]))
    label_columns = first_csv.columns[1:].tolist()  # Skip 'image' column
    num_classes = len(label_columns)
    
    print(f"Number of labels: {num_classes}")
    print(f"Label names: {label_columns[:10]}...")  # Print first 10
    
    # Load all CSV files
    for csv_file in csv_files:
        csv_path = os.path.join(labels_dir, csv_file)
        df = pd.read_csv(csv_path)
        
        class_name = csv_file.replace('.csv', '')
        class_folder = os.path.join(root_dir, class_name)
        
        if not os.path.exists(class_folder):
            continue
        
        for _, row in df.iterrows():
            img_name = row['image']
            img_path = os.path.join(class_folder, img_name)
            
            if os.path.exists(img_path):
                all_image_paths.append(img_path)
                labels = row[1:].values.astype(np.float32)
                all_labels.append(labels)
    
    return all_image_paths, np.array(all_labels), label_columns

# ==================== Novel Components ====================

class ScalePredictor(nn.Module):
    """Predicts importance weights for different scale levels"""
    def __init__(self, in_channels, num_scales=4):
        super().__init__()
        self.num_scales = num_scales
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // 4),
            nn.ReLU(),
            nn.Linear(in_channels // 4, num_scales),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        b, c, h, w = x.shape
        pooled = self.global_pool(x).view(b, c)
        weights = self.fc(pooled)
        return weights

class DynamicScaleAwareFPN(nn.Module):
    """Dynamic Scale-Aware Feature Pyramid"""
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        self.num_scales = len(in_channels_list)
        
        # Lateral connections
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_channels, 1) for in_ch in in_channels_list
        ])
        
        # Scale predictor
        self.scale_predictor = ScalePredictor(in_channels_list[-1], self.num_scales)
        
    def forward(self, features):
        # features: list of feature maps from different scales
        # Get scale weights
        scale_weights = self.scale_predictor(features[-1])
        
        # Apply lateral connections
        lateral_features = [conv(feat) for conv, feat in zip(self.lateral_convs, features)]
        
        # Weighted combination
        # Upsample all to the largest size
        target_size = lateral_features[0].shape[2:]
        upsampled = []
        for feat in lateral_features:
            if feat.shape[2:] != target_size:
                feat = F.interpolate(feat, size=target_size, mode='bilinear', align_corners=False)
            upsampled.append(feat)
        
        # Weight and combine
        weighted_features = []
        for i, feat in enumerate(upsampled):
            weighted_features.append(feat * scale_weights[:, i].view(-1, 1, 1, 1))
        
        output = sum(weighted_features)
        return output, scale_weights

class GraphAttentionLayer(nn.Module):
    """Graph Attention for label correlation"""
    def __init__(self, in_features, out_features, num_labels):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_labels = num_labels
        
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)
        self.leakyrelu = nn.LeakyReLU(0.2)
        
    def forward(self, x, adj=None):
        # x: [batch, num_labels, in_features]
        batch_size = x.size(0)

        Wx = self.W(x)  # [batch, num_labels, out_features]

        # Compute attention coefficients
        a_input = self._prepare_attentional_mechanism_input(Wx)
        # a_input: [batch, num_labels, num_labels, 2*out_features]

        e = self.leakyrelu(self.a(a_input))  # [batch, num_labels, num_labels, 1]
        e = e.squeeze(-1)  # [batch, num_labels, num_labels]

        attention = F.softmax(e, dim=2)  # [batch, num_labels, num_labels]

        # Apply attention
        h_prime = torch.bmm(attention, Wx)  # [batch, num_labels, out_features]

        return h_prime, attention
    
    def _prepare_attentional_mechanism_input(self, Wx):
        num_labels = Wx.size(1)

        # Wx: [batch, num_labels, out_features]
        # We need to create pairs: [batch, num_labels, num_labels, 2*out_features]

        # Repeat for all pairs
        Wx_i = Wx.unsqueeze(2).repeat(1, 1, num_labels, 1)  # [batch, num_labels, num_labels, out_features]
        Wx_j = Wx.unsqueeze(1).repeat(1, num_labels, 1, 1)  # [batch, num_labels, num_labels, out_features]

        # Concatenate pairs
        all_combinations = torch.cat([Wx_i, Wx_j], dim=3)  # [batch, num_labels, num_labels, 2*out_features]

        return all_combinations

class CrossLabelSemanticGraph(nn.Module):
    """Cross-Label Semantic Correlation Graph"""
    def __init__(self, feature_dim, num_labels, num_heads=4):
        super().__init__()
        self.num_labels = num_labels
        self.num_heads = num_heads
        
        self.gat_layers = nn.ModuleList([
            GraphAttentionLayer(feature_dim, feature_dim, num_labels)
            for _ in range(num_heads)
        ])
        
        self.output_projection = nn.Linear(feature_dim * num_heads, feature_dim)
        
    def forward(self, label_features):
        # label_features: [batch, num_labels, feature_dim]
        
        multi_head_outputs = []
        attentions = []
        
        for gat in self.gat_layers:
            out, att = gat(label_features)
            multi_head_outputs.append(out)
            attentions.append(att)
        
        # Concatenate multi-head outputs
        concatenated = torch.cat(multi_head_outputs, dim=2)
        output = self.output_projection(concatenated)
        
        return output, attentions

class DeformableAttention(nn.Module):
    """Memory-efficient Deformable Attention with spatial downsampling"""
    def __init__(self, dim, num_heads=8, max_tokens=196):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.max_tokens = max_tokens  # Reduce to 14x14 = 196 tokens max

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape

        # Spatially downsample if needed to reduce memory
        if N > self.max_tokens:
            # Reshape to 2D, apply adaptive pooling, reshape back
            # Assuming square spatial dimensions
            h = w = int(N ** 0.5)
            target_h = target_w = int(self.max_tokens ** 0.5)

            x_2d = x.transpose(1, 2).reshape(B, C, h, w)
            x_2d = F.adaptive_avg_pool2d(x_2d, (target_h, target_w))
            x = x_2d.reshape(B, C, -1).transpose(1, 2)
            N = x.shape[1]

        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class SpectralSpatialFusion(nn.Module):
    """Spectral-Spatial Decoupled Multi-Head Classification"""
    def __init__(self, feature_dim, num_labels):
        super().__init__()
        
        # Spectral branch
        self.spectral_branch = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(feature_dim // 2, num_labels)
        )
        
        # Spatial branch
        self.spatial_branch = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(feature_dim // 2, num_labels)
        )
        
        # Fusion gate
        self.fusion_gate = nn.Sequential(
            nn.Linear(feature_dim, num_labels),
            nn.Sigmoid()
        )
    
    def forward(self, features):
        spectral_out = self.spectral_branch(features)
        spatial_out = self.spatial_branch(features)
        
        gate = self.fusion_gate(features)
        
        fused_output = gate * spectral_out + (1 - gate) * spatial_out
        
        return fused_output, spectral_out, spatial_out, gate

class UncertaintyHead(nn.Module):
    """Evidential uncertainty estimation"""
    def __init__(self, feature_dim, num_labels):
        super().__init__()
        self.fc = nn.Linear(feature_dim, num_labels)
        
    def forward(self, x):
        evidence = F.softplus(self.fc(x))
        alpha = evidence + 1
        uncertainty = num_labels / torch.sum(alpha, dim=1, keepdim=True)
        return evidence, uncertainty

# ==================== Main AMSI-Net Architecture ====================
class AMSINet(nn.Module):
    def __init__(self, num_labels, pretrained=True):
        super().__init__()
        self.num_labels = num_labels
        
        # Backbone: ResNet50 as feature extractor
        backbone = resnet50(pretrained=pretrained)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        
        self.layer1 = backbone.layer1  # 256 channels
        self.layer2 = backbone.layer2  # 512 channels
        self.layer3 = backbone.layer3  # 1024 channels
        self.layer4 = backbone.layer4  # 2048 channels
        
        # Dynamic Scale-Aware FPN
        self.dsafp = DynamicScaleAwareFPN([256, 512, 1024, 2048], out_channels=256)
        
        # Deformable Attention
        self.deformable_attn = DeformableAttention(dim=256, num_heads=8)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Cross-Label Semantic Correlation Graph
        self.clscg = CrossLabelSemanticGraph(feature_dim=256, num_labels=num_labels, num_heads=4)
        
        # Create label embeddings
        self.label_embeddings = nn.Parameter(torch.randn(num_labels, 256))
        
        # Spectral-Spatial Fusion
        self.spectral_spatial_fusion = SpectralSpatialFusion(256, num_labels)
        
        # Uncertainty Head
        self.uncertainty_head = UncertaintyHead(256, num_labels)
        
    def forward(self, x):
        # Backbone feature extraction
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        # Multi-scale features
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        
        # Dynamic FPN
        fpn_features, scale_weights = self.dsafp([c2, c3, c4, c5])
        
        # Flatten for attention
        b, c, h, w = fpn_features.shape
        fpn_flat = fpn_features.view(b, c, -1).permute(0, 2, 1)  # [B, H*W, C]
        
        # Deformable attention
        attn_features = self.deformable_attn(fpn_flat)
        
        # Global pooling
        attn_features = attn_features.mean(dim=1)  # [B, C]
        
        # Prepare label-specific features
        batch_size = x.size(0)
        label_features = self.label_embeddings.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Add global context to each label
        global_context = attn_features.unsqueeze(1).expand(-1, self.num_labels, -1)
        label_features = label_features + global_context
        
        # Cross-Label Graph reasoning
        graph_features, graph_attentions = self.clscg(label_features)
        
        # Aggregate graph features
        aggregated_features = graph_features.mean(dim=1)  # [B, C]
        
        # Spectral-Spatial Fusion
        output, spectral_out, spatial_out, fusion_gate = self.spectral_spatial_fusion(aggregated_features)
        
        # Uncertainty estimation
        evidence, uncertainty = self.uncertainty_head(aggregated_features)
        
        return {
            'logits': output,
            'spectral_logits': spectral_out,
            'spatial_logits': spatial_out,
            'fusion_gate': fusion_gate,
            'uncertainty': uncertainty,
            'scale_weights': scale_weights,
            'graph_attentions': graph_attentions
        }

# ==================== Loss Functions ====================
class AsymmetricLoss(nn.Module):
    """Asymmetric Loss for multi-label classification"""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        
    def forward(self, x, y):
        # x: logits, y: targets
        xs_pos = torch.sigmoid(x)
        xs_neg = 1 - xs_pos
        
        # Asymmetric Clipping
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
        
        # Basic CE calculation
        los_pos = y * torch.log(xs_pos.clamp(min=1e-8))
        los_neg = (1 - y) * torch.log(xs_neg.clamp(min=1e-8))
        
        # Asymmetric Focusing
        pt0 = xs_pos * y
        pt1 = xs_neg * (1 - y)
        pt = pt0 + pt1
        one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
        one_sided_w = torch.pow(1 - pt, one_sided_gamma)
        
        loss = -one_sided_w * (los_pos + los_neg)
        
        return loss.mean()

class CombinedLoss(nn.Module):
    """Combined loss with multiple components"""
    def __init__(self, num_labels):
        super().__init__()
        self.asymmetric_loss = AsymmetricLoss()
        self.num_labels = num_labels
        
    def forward(self, outputs, targets):
        # Main classification loss
        cls_loss = self.asymmetric_loss(outputs['logits'], targets)
        
        # Spectral and spatial consistency
        spectral_loss = self.asymmetric_loss(outputs['spectral_logits'], targets)
        spatial_loss = self.asymmetric_loss(outputs['spatial_logits'], targets)
        
        # Total loss
        total_loss = cls_loss + 0.3 * (spectral_loss + spatial_loss)
        
        return total_loss

# ==================== Training Functions ====================
def train_epoch(model, dataloader, criterion, optimizer, device, accumulation_steps=2):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    optimizer.zero_grad()

    for batch_idx, (images, targets) in enumerate(tqdm(dataloader, desc="Training")):
        images, targets = images.to(device), targets.to(device)

        outputs = model(images)
        loss = criterion(outputs, targets)

        # Gradient accumulation
        loss = loss / accumulation_steps
        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps

        preds = torch.sigmoid(outputs['logits'])
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(targets.cpu().numpy())

        # Clear cache periodically
        if batch_idx % 100 == 0:
            torch.cuda.empty_cache()

    # Final optimizer step for remaining gradients
    optimizer.step()
    optimizer.zero_grad()

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    # Calculate accuracy (using threshold 0.5)
    pred_labels = (all_preds > 0.5).astype(int)
    accuracy = (pred_labels == all_targets).mean()

    return total_loss / len(dataloader), accuracy

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(tqdm(dataloader, desc="Validation")):
            images, targets = images.to(device), targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            total_loss += loss.item()

            preds = torch.sigmoid(outputs['logits'])
            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

            # Clear cache periodically
            if batch_idx % 100 == 0:
                torch.cuda.empty_cache()

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    pred_labels = (all_preds > 0.5).astype(int)
    accuracy = (pred_labels == all_targets).mean()

    return total_loss / len(dataloader), accuracy, all_preds, all_targets

# ==================== Metrics Calculation ====================
def calculate_metrics(y_true, y_pred, y_scores, label_names):
    """Calculate comprehensive metrics"""
    metrics = {}
    num_labels = y_true.shape[1]
    
    # Per-class metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    
    for i, label in enumerate(label_names):
        tn = ((1 - y_true[:, i]) * (1 - y_pred[:, i])).sum()
        fp = ((1 - y_true[:, i]) * y_pred[:, i]).sum()
        fn = (y_true[:, i] * (1 - y_pred[:, i])).sum()
        tp = (y_true[:, i] * y_pred[:, i]).sum()
        
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
        
        metrics[f'{label}_accuracy'] = accuracy
        metrics[f'{label}_precision'] = precision[i]
        metrics[f'{label}_recall'] = recall[i]
        metrics[f'{label}_f1'] = f1[i]
        metrics[f'{label}_specificity'] = specificity
        
        # MCC
        mcc = matthews_corrcoef(y_true[:, i], y_pred[:, i])
        metrics[f'{label}_mcc'] = mcc
    
    # Overall metrics
    metrics['micro_precision'], metrics['micro_recall'], metrics['micro_f1'], _ = \
        precision_recall_fscore_support(y_true, y_pred, average='micro', zero_division=0)
    
    metrics['macro_precision'], metrics['macro_recall'], metrics['macro_f1'], _ = \
        precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    
    metrics['weighted_precision'], metrics['weighted_recall'], metrics['weighted_f1'], _ = \
        precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    
    # Hamming metrics
    metrics['hamming_accuracy'] = 1 - (np.sum(y_pred != y_true) / (y_true.shape[0] * y_true.shape[1]))
    
    # ROC-AUC
    try:
        metrics['micro_roc_auc'] = roc_auc_score(y_true, y_scores, average='micro')
        metrics['macro_roc_auc'] = roc_auc_score(y_true, y_scores, average='macro')
    except:
        metrics['micro_roc_auc'] = 0
        metrics['macro_roc_auc'] = 0
    
    # Average Precision
    try:
        metrics['micro_map'] = average_precision_score(y_true, y_scores, average='micro')
        metrics['macro_map'] = average_precision_score(y_true, y_scores, average='macro')
    except:
        metrics['micro_map'] = 0
        metrics['macro_map'] = 0
    
    # LRAP
    try:
        metrics['lrap'] = label_ranking_average_precision_score(y_true, y_scores)
    except:
        metrics['lrap'] = 0
    
    return metrics

def calculate_ece(y_true, y_scores, n_bins=10):
    """Calculate Expected Calibration Error"""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0
    
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        
        in_bin = (y_scores > bin_lower) & (y_scores <= bin_upper)
        prop_in_bin = in_bin.mean()
        
        if prop_in_bin > 0:
            accuracy_in_bin = (y_true[in_bin] == (y_scores[in_bin] > 0.5)).mean()
            avg_confidence_in_bin = y_scores[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    
    return ece

def calculate_brier_score(y_true, y_scores):
    """Calculate Brier Score"""
    return np.mean((y_scores - y_true) ** 2)

# ==================== Main Execution ====================
if __name__ == "__main__":
    # Configuration
    ROOT_DIR = "/home/aiserver/Desktop/Remote-Sensing-Datasets/MLRSNet-master/image"
    LABELS_DIR = "/home/aiserver/Desktop/Remote-Sensing-Datasets/MLRSNet-master/labels"
    BATCH_SIZE = 32  # Reduced from 32 to save memory
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Enable memory optimization
    torch.backends.cudnn.benchmark = True
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print(f"Using device: {DEVICE}")
    
    # Data transforms
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Load dataset
    print("Loading dataset...")
    image_paths, labels, label_names = load_dataset(ROOT_DIR, LABELS_DIR)
    print(f"Total samples: {len(image_paths)}")
    
    # Split dataset: 80% train, 10% val, 10% test
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.2, random_state=42
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42
    )
    
    print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")
    
    # Create datasets
    train_dataset = MultiLabelDataset(train_paths, train_labels, train_transform)
    val_dataset = MultiLabelDataset(val_paths, val_labels, val_transform)
    test_dataset = MultiLabelDataset(test_paths, test_labels, val_transform)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    # Initialize model
    num_labels = labels.shape[1]
    model = AMSINet(num_labels=num_labels, pretrained=True).to(DEVICE)
    
    # Loss and optimizer
    criterion = CombinedLoss(num_labels)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # Training
    print("\nStarting training...")
    best_val_acc = 0
    patience = 15
    patience_counter = 0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_preds, val_targets = validate(model, val_loader, criterion, DEVICE)
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
        scheduler.step(val_acc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"Best model saved with validation accuracy: {best_val_acc:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Plot training curves
    plt.figure(figsize=(12, 5), dpi=600)
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.plot(val_losses, label='Val Loss', linewidth=2)
    plt.xlabel('Epoch', fontweight='bold', fontsize=12)
    plt.ylabel('Loss', fontweight='bold', fontsize=12)
    plt.title('Training and Validation Loss', fontweight='bold', fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Accuracy', linewidth=2)
    plt.plot(val_accs, label='Val Accuracy', linewidth=2)
    plt.xlabel('Epoch', fontweight='bold', fontsize=12)
    plt.ylabel('Accuracy', fontweight='bold', fontsize=12)
    plt.title('Training and Validation Accuracy', fontweight='bold', fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    # Load best model for testing
    print("\nLoading best model for testing...")
    model.load_state_dict(torch.load('best_model.pth'))
    
    # Testing
    print("\nEvaluating on test set...")
    test_loss, test_acc, test_preds, test_targets = validate(model, test_loader, criterion, DEVICE)
    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    # Convert predictions to labels
    test_pred_labels = (test_preds > 0.5).astype(int)
    
    # Calculate comprehensive metrics
    print("\nCalculating metrics...")
    metrics = calculate_metrics(test_targets, test_pred_labels, test_preds, label_names)
    
    # Save detailed results
    with open('results.txt', 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("AMSI-Net Test Results\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("Overall Metrics:\n")
        f.write("-" * 80 + "\n")
        f.write(f"Test Accuracy: {test_acc:.4f}\n")
        f.write(f"Micro Precision: {metrics['micro_precision']:.4f}\n")
        f.write(f"Micro Recall: {metrics['micro_recall']:.4f}\n")
        f.write(f"Micro F1: {metrics['micro_f1']:.4f}\n")
        f.write(f"Macro Precision: {metrics['macro_precision']:.4f}\n")
        f.write(f"Macro Recall: {metrics['macro_recall']:.4f}\n")
        f.write(f"Macro F1: {metrics['macro_f1']:.4f}\n")
        f.write(f"Weighted Precision: {metrics['weighted_precision']:.4f}\n")
        f.write(f"Weighted Recall: {metrics['weighted_recall']:.4f}\n")
        f.write(f"Weighted F1: {metrics['weighted_f1']:.4f}\n")
        f.write(f"Hamming Accuracy: {metrics['hamming_accuracy']:.4f}\n")
        f.write(f"Micro ROC-AUC: {metrics['micro_roc_auc']:.4f}\n")
        f.write(f"Macro ROC-AUC: {metrics['macro_roc_auc']:.4f}\n")
        f.write(f"Micro mAP: {metrics['micro_map']:.4f}\n")
        f.write(f"Macro mAP: {metrics['macro_map']:.4f}\n")
        f.write(f"LRAP: {metrics['lrap']:.4f}\n\n")
        
        # Per-class metrics
        f.write("Per-Class Metrics:\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Label':<25} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'Spec':>8} {'MCC':>8}\n")
        f.write("-" * 80 + "\n")
        
        for label in label_names:
            f.write(f"{label:<25} "
                   f"{metrics[f'{label}_accuracy']:>8.4f} "
                   f"{metrics[f'{label}_precision']:>8.4f} "
                   f"{metrics[f'{label}_recall']:>8.4f} "
                   f"{metrics[f'{label}_f1']:>8.4f} "
                   f"{metrics[f'{label}_specificity']:>8.4f} "
                   f"{metrics[f'{label}_mcc']:>8.4f}\n")
    
    # ECE and Brier Score per class
    print("Calculating ECE and Brier Score...")
    ece_scores = []
    brier_scores = []
    
    for i in range(num_labels):
        ece = calculate_ece(test_targets[:, i], test_preds[:, i])
        brier = calculate_brier_score(test_targets[:, i], test_preds[:, i])
        ece_scores.append(ece)
        brier_scores.append(brier)
    
    with open('results.txt', 'a') as f:
        f.write("\n\nCalibration Metrics:\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Label':<25} {'ECE':>10} {'Brier':>10}\n")
        f.write("-" * 80 + "\n")
        for i, label in enumerate(label_names):
            f.write(f"{label:<25} {ece_scores[i]:>10.4f} {brier_scores[i]:>10.4f}\n")
    
    # Confusion matrices for each label
    print("Generating confusion matrices...")
    fig, axes = plt.subplots(int(np.ceil(num_labels/5)), 5, figsize=(25, 5*int(np.ceil(num_labels/5))), dpi=600)
    axes = axes.flatten() if num_labels > 1 else [axes]
    
    for i, label in enumerate(label_names[:min(num_labels, len(axes))]):
        cm = confusion_matrix(test_targets[:, i], test_pred_labels[:, i])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], 
                   annot_kws={'size': 14, 'weight': 'bold'},
                   cbar_kws={'label': 'Count'})
        axes[i].set_title(label, fontweight='bold', fontsize=12)
        axes[i].set_xlabel('Predicted', fontweight='bold', fontsize=10)
        axes[i].set_ylabel('Actual', fontweight='bold', fontsize=10)
    
    # Hide unused subplots
    for i in range(num_labels, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    # ROC Curves
    print("Generating ROC curves...")
    plt.figure(figsize=(15, 10), dpi=600)
    
    for i, label in enumerate(label_names[:min(20, num_labels)]):  # Plot first 20 to avoid clutter
        try:
            fpr, tpr, _ = roc_curve(test_targets[:, i], test_preds[:, i])
            roc_auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
            plt.plot(fpr, tpr, linewidth=2, label=f'{label} (AUC={roc_auc:.3f})', alpha=0.7)
        except:
            continue
    
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
    plt.xlabel('False Positive Rate', fontweight='bold', fontsize=14)
    plt.ylabel('True Positive Rate', fontweight='bold', fontsize=14)
    plt.title('ROC Curves', fontweight='bold', fontsize=16)
    plt.legend(loc='lower right', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    # Precision-Recall Curves
    print("Generating PR curves...")
    plt.figure(figsize=(15, 10), dpi=600)
    
    for i, label in enumerate(label_names[:min(20, num_labels)]):
        try:
            precision, recall, _ = precision_recall_curve(test_targets[:, i], test_preds[:, i])
            ap = average_precision_score(test_targets[:, i], test_preds[:, i])
            plt.plot(recall, precision, linewidth=2, label=f'{label} (AP={ap:.3f})', alpha=0.7)
        except:
            continue
    
    plt.xlabel('Recall', fontweight='bold', fontsize=14)
    plt.ylabel('Precision', fontweight='bold', fontsize=14)
    plt.title('Precision-Recall Curves', fontweight='bold', fontsize=16)
    plt.legend(loc='best', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pr_curves.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    # Calibration curve
    print("Generating calibration curve...")
    plt.figure(figsize=(10, 8), dpi=600)
    
    n_bins = 10
    for i in range(min(5, num_labels)):  # Show first 5 labels
        fraction_of_positives, mean_predicted_value = [], []
        
        for j in range(n_bins):
            bin_lower = j / n_bins
            bin_upper = (j + 1) / n_bins
            
            in_bin = (test_preds[:, i] > bin_lower) & (test_preds[:, i] <= bin_upper)
            
            if in_bin.sum() > 0:
                fraction_of_positives.append(test_targets[:, i][in_bin].mean())
                mean_predicted_value.append(test_preds[:, i][in_bin].mean())
        
        plt.plot(mean_predicted_value, fraction_of_positives, 'o-', 
                linewidth=2, markersize=8, label=label_names[i], alpha=0.7)
    
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
    plt.xlabel('Mean Predicted Probability', fontweight='bold', fontsize=14)
    plt.ylabel('Fraction of Positives', fontweight='bold', fontsize=14)
    plt.title('Calibration Curve', fontweight='bold', fontsize=16)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('calibration_curve.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    # Calculate number of parameters and FLOPs (approximation)
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    num_params = count_parameters(model)
    print(f"\nNumber of trainable parameters: {num_params:,}")
    
    with open('results.txt', 'a') as f:
        f.write(f"\n\nModel Statistics:\n")
        f.write("-" * 80 + "\n")
        f.write(f"Trainable Parameters: {num_params:,}\n")
    
    # Misclassification analysis
    print("Generating misclassification analysis...")
    misclass_per_label = []
    for i in range(num_labels):
        misclass = (test_pred_labels[:, i] != test_targets[:, i]).sum()
        misclass_per_label.append(misclass)
    
    plt.figure(figsize=(20, 6), dpi=600)
    plt.bar(range(min(30, num_labels)), misclass_per_label[:30], color='coral', edgecolor='black')
    plt.xlabel('Label Index', fontweight='bold', fontsize=14)
    plt.ylabel('Number of Misclassifications', fontweight='bold', fontsize=14)
    plt.title('Misclassification Analysis', fontweight='bold', fontsize=16)
    plt.xticks(range(min(30, num_labels)), [label_names[i][:10] for i in range(min(30, num_labels))], 
              rotation=45, ha='right', fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('misclassification_analysis.png', dpi=600, bbox_inches='tight')
    plt.close()
    
    # Confusion pairs analysis
    print("Analyzing confused label pairs...")
    confusion_pairs = []
    
    for i in range(num_labels):
        for j in range(i+1, num_labels):
            # Cases where both are wrong in opposite directions
            confusion_count = ((test_targets[:, i] == 1) & (test_pred_labels[:, i] == 0) & 
                             (test_targets[:, j] == 0) & (test_pred_labels[:, j] == 1)).sum()
            confusion_count += ((test_targets[:, i] == 0) & (test_pred_labels[:, i] == 1) & 
                              (test_targets[:, j] == 1) & (test_pred_labels[:, j] == 0)).sum()
            
            if confusion_count > 0:
                confusion_pairs.append((label_names[i], label_names[j], confusion_count))
    
    # Sort and get top 10
    confusion_pairs.sort(key=lambda x: x[2], reverse=True)
    top_confused = confusion_pairs[:10]
    
    if top_confused:
        plt.figure(figsize=(12, 8), dpi=600)
        labels = [f"{p[0][:8]}\nvs\n{p[1][:8]}" for p in top_confused]
        counts = [p[2] for p in top_confused]
        
        plt.barh(range(len(top_confused)), counts, color='steelblue', edgecolor='black')
        plt.yticks(range(len(top_confused)), labels, fontweight='bold', fontsize=10)
        plt.xlabel('Confusion Count', fontweight='bold', fontsize=14)
        plt.title('Top 10 Most Confused Label Pairs', fontweight='bold', fontsize=16)
        plt.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.savefig('confused_pairs.png', dpi=600, bbox_inches='tight')
        plt.close()
    
    print("\n" + "="*80)
    print("Training and evaluation complete!")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print("All results and figures saved.")
    print("="*80)

Using device: cuda
Loading dataset...
Number of labels: 60
Label names: ['airplane', 'airport', 'bare soil', 'baseball diamond', 'basketball court', 'beach', 'bridge', 'buildings', 'cars', 'chaparral']...
Total samples: 109161
Train: 87328, Val: 10916, Test: 10917

Starting training...

Epoch 1/100


Validation: 100%|██████████| 342/342 [00:21<00:00, 15.95it/s]


Train Loss: 0.0363, Train Acc: 0.9280
Val Loss: 0.0217, Val Acc: 0.9609
Best model saved with validation accuracy: 0.9609

Epoch 2/100


Validation: 100%|██████████| 342/342 [00:20<00:00, 16.42it/s]


Train Loss: 0.0240, Train Acc: 0.9566
Val Loss: 0.0193, Val Acc: 0.9644
Best model saved with validation accuracy: 0.9644

Epoch 3/100


...

Train Loss: 0.0008, Train Acc: 0.9989
Val Loss: 0.0355, Val Acc: 0.9845

Epoch 100/100


Validation: 100%|██████████| 342/342 [00:20<00:00, 16.43it/s]


Train Loss: 0.0008, Train Acc: 0.9989
Val Loss: 0.0355, Val Acc: 0.9845

Loading best model for testing...

Evaluating on test set...


Validation: 100%|██████████| 342/342 [00:21<00:00, 15.69it/s]


Test Loss: 0.0375, Test Acc: 0.9843

Calculating metrics...
Calculating ECE and Brier Score...
Generating confusion matrices...
Generating ROC curves...
Generating PR curves...
Generating calibration curve...

Number of trainable parameters: 26,460,468
Generating misclassification analysis...
Analyzing confused label pairs...

Training and evaluation complete!
Best validation accuracy: 0.9846
Test accuracy: 0.9843
All results and figures saved.
